# Algorithms 23-25: Regression via Pseudoinverse**Learning Objectives:**1. Formulate curve-fitting as an overdetermined system of linear equations $y = Xp$2. Use the Pseudoinverse $(X^T X)^{-1} X^T$ to find the optimal least-squares parameters $p$3. Extend the pseudoinverse to fit Polynomial, Multiple Linear, Power, Exponential, and Logistic models via algebraic transformation**Prerequisites:** Algorithms 15 (RREF/Matrix Inversion)**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapters 23-25

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "23-25_regression_pseudoinverse",    "level": 3,    "title": "Algorithms 23-25: Regression via Pseudoinverse",    "prerequisites": [        "Algorithms 15 (RREF/Matrix Inversion)"    ],    "skills_taught": [        "lesson.algorithms_23_25_regression_pseudoinverse"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "26_overfitting_bias_variance.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_23_25_regression_pseudoinverse",        "title": "Lesson Algorithms 23 25 Regression Pseudoinverse",        "notebook": "23-25_regression_pseudoinverse.ipynb",        "level": 3    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "23-25_regression_pseudoinverse.ipynb",        "level": 3    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Algorithms 15 (RREF/Matrix Inversion).

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_23_25_regression_pseudoinverse', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setupimport mathimport matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### The Supervised Learning Problem> 📖 *From Algorithms Ch. 23:* Supervised learning consists of fitting functions to data sets. We want to come up with a mathematical model that predicts outputs for other inputs.Given data `[(0, 1), (2, 5), (4, 3)]`, we want to fit a line $y = mx + b$. We can plug each point into the equation:$1 = m(0) + b$$5 = m(2) + b$$3 = m(4) + b$This is an overdetermined system (3 equations, 2 unknowns). There is no perfect solution. In matrix form, $y = Xp$:$\begin{bmatrix} 1 \\ 5 \\ 3 \end{bmatrix} = \begin{bmatrix} 0 & 1 \\ 2 & 1 \\ 4 & 1 \end{bmatrix} \begin{bmatrix} m \\ b \end{bmatrix}$### The PseudoinverseBecause $X$ is tall and rectangular, it has no inverse. But if we multiply both sides by $X^T$, we project the problem into a square matrix space!$X^T y = X^T X p$Now, $X^T X$ is a square matrix (in this case $2 \times 2$). If its columns are independent, it is invertible!$p = (X^T X)^{-1} X^T y$This magical formula guarantees the mathematically optimal least-squares fit for *any* linear combination of functions!

### Comprehension Check ✅How would you set up the $X$ matrix to fit a quadratic model $y = ax^2 + bx + c$ to the same three points?<details><summary>💡 Explanation after a retrieval attempt</summary>Each row in $X$ would have 3 columns: $x^2$, $x$, and $1$.$\begin{bmatrix} 1 \\ 5 \\ 3 \end{bmatrix} = \begin{bmatrix} 0^2 & 0 & 1 \\ 2^2 & 2 & 1 \\ 4^2 & 4 & 1 \end{bmatrix} \begin{bmatrix} a \\ b \\ c \end{bmatrix}$</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: Non-Linear TransformationsThe pseudoinverse can only solve models of the form $y = c_1 f_1(x) + c_2 f_2(x) + \dots$.If we want to fit a **Power Regression** $y = a \cdot x^b$, we must transform it.Take the natural logarithm of both sides and expand the right side using logarithm rules.What does the equation look like now? What are the new "variables" and "constants"?<details><summary>💡 Explanation after a retrieval attempt</summary>$\ln(y) = \ln(a \cdot x^b)$$\ln(y) = \ln(a) + b \ln(x)$Let $Y = \ln(y)$ and $X_{new} = \ln(x)$. The equation is $Y = b X_{new} + \ln(a)$, which is a straight line! We can solve for $b$ and $\ln(a)$ using the pseudoinverse on the transformed data.</details>

---## Phase 1 — Algorithm Construction### Step 1: The Matrix Class (Refresher)To do this, we need matrix multiplication, transpose, and inverse. Bring over your `Matrix` class from Algorithms 15. *(For the sake of time in this lab, you may use a simplified list-of-lists approach or a basic pure-Python matrix library if you already built one. Here is a starter.)*

In [ ]:
class Matrix:    def __init__(self, data):        self.data = data        self.rows = len(data)        self.cols = len(data[0])            def transpose(self):        return Matrix([[self.data[r][c] for r in range(self.rows)] for c in range(self.cols)])            def multiply(self, other):        result = [[sum(self.data[r][k] * other.data[k][c] for k in range(self.cols))                    for c in range(other.cols)] for r in range(self.rows)]        return Matrix(result)            # We need an inverse method! For 2x2, there is a simple formula:    def inverse_2x2(self):        a, b = self.data[0][0], self.data[0][1]        c, d = self.data[1][0], self.data[1][1]        det = a*d - b*c        return Matrix([[d/det, -b/det], [-c/det, a/det]])

### Step 2: Linear Regression SolverWrite a function `fit_linear(points)` that takes a list of `(x, y)` tuples and returns `(m, b)`.

In [ ]:
def fit_linear(points):    # 1. Construct X and Y matrices    # X should be [[x1, 1], [x2, 1], ...]    # Y should be [[y1], [y2], ...]    # YOUR CODE HERE        # 2. Compute X_T = X.transpose()    # 3. Compute X_T_X = X_T.multiply(X)    # 4. Compute X_T_X_inv = X_T_X.inverse_2x2()    # 5. Compute X_T_Y = X_T.multiply(Y)    # 6. Compute p = X_T_X_inv.multiply(X_T_Y)    # return (p.data[0][0], p.data[1][0]) # (m, b)    pass

### Step 3: Exponential Regression SolverWrite a function `fit_exponential(points)` that fits $y = a \cdot b^x$.Transform the data, use `fit_linear`, and then reverse the transformation to find $a$ and $b$.

In [ ]:
def fit_exponential(points):    # y = a * b^x  =>  ln(y) = ln(a) + x * ln(b)    # This is a line: Y = m*X + B, where Y = ln(y), m = ln(b), X = x, B = ln(a)        # 1. Transform the points: (x, y) -> (x, ln(y))    # YOUR CODE HERE        # 2. Call fit_linear on the transformed points to get (m, B)    # YOUR CODE HERE        # 3. Reverse the transformation:    # b = math.exp(m)    # a = math.exp(B)    # return (a, b)    pass

---## Phase 2 — Experimental Verification### Fitting and PlottingTest your algorithms on the dataset `[(1, 2), (2, 4), (3, 7), (4, 15), (5, 30)]`.Plot the data points as a scatter plot. Then, generate $X$ values from 1 to 5, compute the predicted $Y$ values using your exponential model, and plot the curve!

In [ ]:
# data = [(1, 2), (2, 4), (3, 7), (4, 15), (5, 30)]# a, b = fit_exponential(data)# print(f"Model: y = {a:.3f} * {b:.3f}^x")# xs = [p[0] for p in data]# ys = [p[1] for p in data]# plt.scatter(xs, ys, color='red', label="Data")# plot_xs = [1 + i*0.1 for i in range(41)]# plot_ys = [a * (b**x) for x in plot_xs]# plt.plot(plot_xs, plot_ys, color='blue', label="Exponential Fit")# plt.legend(); plt.grid(True); plt.show()

---## Phase 3 — Olympiad Extension### The Danger of Transformation> 📖 *From Algorithms Ch. 25:* If we transform $y = a x^b$ into $\ln(y) = \ln(a) + b \ln(x)$, the pseudoinverse finds the absolute mathematically optimal fit for the *transformed* equation. It does **NOT** guarantee the optimal fit for the *original* equation!This is because distances are warped by the logarithm. An error of $100$ at $y=1000$ looks like a tiny error in $\ln(y)$ space, while an error of $1$ at $y=2$ looks huge in $\ln(y)$ space! How could we solve for the parameters of $y = a x^b$ WITHOUT transforming the equation, ensuring we get the true optimal fit in the original space?<details><summary>💡 Sketch after a retrieval attempt</summary>We cannot use the exact algebraic pseudoinverse formula, because the equation is strictly non-linear with respect to its parameters. Instead, we must formulate an error function $E(a, b) = \sum (a x_i^b - y_i)^2$, compute the partial derivatives $\frac{\partial E}{\partial a}$ and $\frac{\partial E}{\partial b}$, and use **Gradient Descent** (Chapter 27) to walk down the 3D error surface until we find the minimum!</details>---📚 **Next:** Algorithms 26 (K-Nearest Neighbors)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_23_25_regression_pseudoinverse', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='26_overfitting_bias_variance.ipynb')